# Validação da Co-simulação de Veículos Elétricos
Este notebook executa o cenário de co-simulação `scenario_ESB01S4_EVs.py` e realiza a análise do comportamento dinâmico dos carregadores (EVSEs) acoplados fisicamente à rede elétrica via Mosaik e Caldera ICM.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import subprocess
import time
from pathlib import Path
import os
import matplotlib.dates as mdates

## 1. Execução da Co-simulação

In [ ]:
# Caminho absoluto do script e da pasta do projeto
PROJECT_ROOT = Path(os.getcwd()).parent
SCRIPT_PATH = PROJECT_ROOT / 'scenarios' / 'scenario_ESB01S4_EVs.py'

print("===========================================================")
print(" Iniciando Co-simulação Mosaik (OpenDSS + PV + Caldera EV) ")
print("===========================================================")

start_time = time.time()

# Executa o cenário como subprocesso usando uv run
result = subprocess.run(
    ["uv", "run", "python", str(SCRIPT_PATH)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

end_time = time.time()
print(f"Tempo de execução da simulação: {end_time - start_time:.2f} segundos\n")

# Exibir ultimas linhas do log do console
lines = result.stdout.split('\n')
for line in lines[-20:]:
    print(line)

if result.returncode != 0:
    print("\n[ERRO NA SIMULAÇÃO]:\n", result.stderr)

## 2. Leitura e Processamento dos Resultados

In [ ]:
CSV_FILE = PROJECT_ROOT / 'output' / 'result_run_ESB01S4_EVs.csv'

print(f"Lendo resultados do arquivo: {CSV_FILE.name}...")
df = pd.read_csv(CSV_FILE)
df['date'] = pd.to_datetime(df['date'], format='mixed')
df.set_index('date', inplace=True)

print(f"Linhas totais: {len(df)}")
print(f"Número de colunas coletadas: {len(df.columns)}")

## 3. Relatório do Terminal: Demanda Ativa e Fator de Potência dos VEs

In [ ]:
# Filtrando as colunas relacionadas a Veículos Elétricos
ev_p_cols = [c for c in df.columns if 'CalderaEV' in c and 'P_kw' in c]
ev_q_cols = [c for c in df.columns if 'CalderaEV' in c and 'Q_kvar' in c]
ev_fp_cols = [c for c in df.columns if 'CalderaEV' in c and 'fp' in c]

df['Total_EV_P_kw'] = df[ev_p_cols].sum(axis=1)
df['Total_EV_Q_kvar'] = df[ev_q_cols].sum(axis=1)

max_p_kw = df['Total_EV_P_kw'].max()
time_max_p = df['Total_EV_P_kw'].idxmax()

step_size_h = 1.0 / 60.0 # 1 minuto = 1/60 horas
energia_total_kwh = df['Total_EV_P_kw'].sum() * step_size_h

print("="*60)
print("           RELATÓRIO DE COMPORTAMENTO DOS VEs")
print("="*60)
print(f"Total de VEs simulados: {len(ev_p_cols)}")
print(f"Energia total demandada na semana: {energia_total_kwh:.2f} kWh")
print(f"Pico de Demanda Agregada dos VEs: {max_p_kw:.2f} kW")
print(f"Ocorrência do Pico: {time_max_p.strftime('%d/%m/%Y %H:%M')}")

# Analisando o Fator de Potência (PFC)
mean_fps = df[ev_fp_cols].replace(0, np.nan).mean().mean()
print(f"Fator de Potência médio em operação (Active PFC): {mean_fps:.4f}")
print("="*60)

## 4. Análise Gráfica

In [ ]:
plt.style.use('default')
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df.index, df['Total_EV_P_kw'], color='#FF5722', linewidth=1.5, label='Demanda Ativa VEs (P_kw)')
ax.plot(df.index, df['Total_EV_Q_kvar'], color='#03A9F4', linewidth=1.5, label='Demanda Reativa VEs (Q_kvar)', alpha=0.8)

ax.set_title('Perfil de Demanda Agregada de Todos os Veículos Elétricos', fontsize=14, fontweight='bold')
ax.set_ylabel('Potência (kW / kVAr)', fontsize=12)
ax.set_xlabel('Tempo', fontsize=12)
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.6)

# Formatando eixo x
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Selecionar os 3 VEs que consumiram mais energia para evidenciar o Tapering (CC-CV)
top_3_evs = df[ev_p_cols].sum().nlargest(3).index

for i, col in enumerate(top_3_evs):
    ax.plot(df.index, df[col], linewidth=2, label=f'Veículo {i+1} (Wallbox)')

ax.set_title('Assinatura Física Individual (Evidência do Efeito Tapering CV)', fontsize=14, fontweight='bold')
ax.set_ylabel('Potência Ativa (kW)', fontsize=12)
ax.set_xlabel('Tempo', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %H:%M'))
plt.xticks(rotation=45)

# Dando zoom nas primeiras 72 horas para ficar mais visível
ax.set_xlim(df.index[0], df.index[int(72*60)]) 

plt.tight_layout()
plt.show()

In [ ]:
# Tensão na Rede (Pegando a primeira barra de teste logada)
v1_cols = [c for c in df.columns if 'V1_pu' in c]
if len(v1_cols) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Plot tensão da barra 0
    ax.plot(df.index, df[v1_cols[0]], color='purple', linewidth=1)
    
    ax.set_title('Impacto na Tensão da Rede (Fase 1 - p.u.)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Tensão (pu)', fontsize=12)
    ax.set_xlabel('Tempo', fontsize=12)
    ax.axhline(0.93, color='red', linestyle='--', label='Limite Mínimo Precário (0.93 pu)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %d/%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Colunas de tensão não encontradas no dataset atual.")